# C6 · Sustracción de halo LPM

**Spec:** [`docs/spec_C6_codex_lpm_subtraction.md`](../docs/spec_C6_codex_lpm_subtraction.md)  |  **Bloque:** C · Extracción  |  **Run de este set:** `ROXs42Bb_realigned`

Sustrae el halo estelar modelando cada spaxel como modulación polinomial de Legendre (grado 4) del espectro de referencia, con las líneas de ciencia enmascaradas del ajuste (Julo et al. 2025 App. A.4).

| | |
|---|---|
| **Entrada** | `stage02` stack + posiciones (B3) + PSF (C1, solo apcorr) |
| **Salida (QC/productos)** | `stages/spec_lpm_qc.json`, `spec_lpm_object.fits`, cubo residual, mapas de coeficientes |
| **Consume aguas abajo** | D1 v3, G1, E4; cubo residual → E1b (mapa FoV) |


## Qué hace C6 y por qué preserva las líneas

C6 implementa el método **propuesto** por Julo et al. 2025 (LPM): cada spaxel se modela como `ŝ_xy = Σ_k β_k · P_k(λ̃) · ŝ` (Legendre de grado 4 modulando la referencia), resuelto por mínimos cuadrados **con las líneas de ciencia fuera del ajuste**. El modelo interpola suavemente a través de las líneas → el flujo y el perfil del compañero sobreviven (sin auto-sustracción estructural), y el continuo vecino no se hunde.

**Diagnósticos de grado (QC, nunca ajuste al vuelo):** energy-share por grado (Fig. 8 del paper), curva MSE analítica descompuesta en underfit-estrella / overfit-planeta / overfit-ruido (Fig. 6 / Ec. B.5) y mapas de coeficientes que descomponen la PSF por frecuencia espectral (Fig. 7: radio AO, spikes, anillos de Airy). Si señalan que ∂=4 no basta, se revisa la spec — el grado no cambia dentro del run.

Límite conocido (paper §4.1): la componente del espectro planetario **colineal** con la referencia (p.ej. su continuo) se absorbe en el modelo; el LPM es óptimo para compañeras dominadas por líneas.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs42Bb_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
python -m musepipe.stages.stage_x05_lpm --run-id $RUN
```

Ligero (~1–2 min: una pseudo-inversa compartida + diagnósticos de grado).

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/spec_lpm_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python -m musepipe.stages.stage_x05_lpm --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/spec_lpm_qc.json', RUN_ID)
nb.show(qc, keys=['lpm.degree', 'lpm.line_preservation_recovery', 'degree_diagnostics.mse_argmin', 'checks.v2_line_preservation_ok', 'checks.v4_slow_path_ok', 'checks.v5_scale_convention_ok'], title='C6')


## Los chequeos del QC, en físico

| Chequeo | ¿Qué pregunta contesta? | Si falla |
|---|---|---|
| `v1_reference_ok` | **¿Hay bastante halo para la referencia?** ≥ 50 spaxels por exposición, como en C5. | La referencia es ruido. |
| `v2_line_preservation_ok` | **¿El método respeta la línea, que es lo que buscamos?** Se inyecta una gaussiana de 5σ en un control, se sustrae con la misma matriz y se exige recuperar **≥ 90%** del flujo. Es el argumento de LPM frente a SGF, que sí muerde la línea. | El método estaría borrando la señal de acreción junto con el halo. Es un smoke interno: la calibración formal es E4. |
| `v3_condition_ok` | **¿El ajuste es numéricamente estable?** Número de condición < 1e8. | Los coeficientes son ruido amplificado: el residuo deja de significar nada. |
| `v4_slow_path_ok` | **¿Cuántos spaxels necesitaron el camino lento?** ≤ 20%. | Señal de mal condicionamiento generalizado (y de coste). |
| `v5_scale_convention_ok` | **¿Escala y cabeceras trazables?** Igual que C5. | D1 no puede comparar sin adivinar la convención. |
| `v6_degree_diagnostics_written` | **¿Está justificado el grado 4 del polinomio?** Energy-share, curva de MSE y mapas de coeficientes persistidos. | Se pierde la evidencia de por qué ese grado y no otro (avisos que no bloquean: se miran en los plots). |


## Evidencia: preservación de línea y diagnósticos de grado


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('C6', 'stages/spec_lpm_qc.json'):
        q = nb.load_qc('stages/spec_lpm_qc.json', RUN_ID)
        l = q['lpm']; d = q['degree_diagnostics']
        print(f"grado={l['degree']}  máscara={l['masked_lines_A']}")
        print(f"condición={max(l['condition_number']):.2e}  slow_frac={l['slow_fraction_max']:.3f}")
        print(f"smoke de preservación de línea (control): {l['line_preservation_recovery']}")
        print(f"energy-share (g1..g9): {[f'{v:.3f}' for v in d['energy_share']]}")
        print(f"MSE argmin={d['mse_argmin']}  warns: grado={d['degree_check_warn']} mse={d['mse_check_warn']}")
        print('checks:', q['checks'])


## Plot 1 — mapas de coeficientes (Fig. 7 del paper)

Planos β̂_k del ajuste diagnóstico de grado 9 (primera exposición): los grados bajos muestran el radio AO y los spikes; los altos, anillos de Airy y ruido.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    q = nb.load_qc('stages/spec_lpm_qc.json', RUN_ID)
    with fits.open(q['products']['coeff_maps']) as h:
        maps = np.asarray(h['COEFFS_DEG9'].data, float)
    fig, axes = plt.subplots(2, 5, figsize=(14, 5.6))
    for k, ax in enumerate(axes.ravel()):
        m = maps[k]
        v = np.nanpercentile(m, [25, 75])
        ax.imshow(m, origin='lower', cmap='RdBu_r', vmin=v[0], vmax=v[1])
        ax.set_title(f'grado {k}', fontsize=9); ax.axis('off')
    fig.suptitle('C6 · mapas de coeficientes LPM (descomposición de la PSF)')
    fig.tight_layout(); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — SGF vs LPM alrededor de Hα (Fig. 13/14 del paper)

Los dos espectros del compañero en ±80 Å de Hα: si hay línea, el SGF la auto-sustrae y hunde el continuo vecino; el LPM la preserva. Requiere C5 corrido.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    rd = nb.run_dir(RUN_ID)
    def spec(name):
        h = fits.open(rd / 'stages' / name)
        w = np.asarray(h[1].data['wave_A'], float); f = np.asarray(h[1].data['flux'], float)
        h.close(); return w, f
    w_s, f_s = spec('spec_sgf_object.fits')
    w_l, f_l = spec('spec_lpm_object.fits')
    sel = np.abs(w_s - 6563.0) <= 80
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(w_s[sel], f_s[sel], lw=1.0, color='tab:blue', label='SGF (C5)')
    ax.plot(w_l[sel], f_l[sel], lw=1.0, color='tab:orange', label='LPM (C6)')
    ax.axvline(6563, color='tab:red', ls=':'); ax.axhline(0, color='0.6', lw=0.6)
    ax.set_xlabel('λ [Å]'); ax.set_title('C6 · SGF vs LPM alrededor de Hα')
    ax.legend(); fig.tight_layout(); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Figura de paper — el espectro sin binar, con su error y sus líneas

Las figuras anteriores son de diagnóstico. Ésta es la que se publica, y por eso cambia en tres cosas:

- **Sin binar**: cada canal con su σ. Binar es cómodo para leer un continuo, pero esconde justo lo que se quiere enseñar (o no enseñar): que en Hα no hay nada por encima del ruido **a la resolución del dato**.
- **Dos barras de error**: la **empírica** (dispersión de los controles procesados igual que el objeto) como banda, y la **propagada del STAT** como línea. Que se vean las dos es la forma honesta de enseñar que el STAT del cubo no es σ ([`docs/noise_model.md`](../docs/noise_model.md)).
- **Marcado completo**: las **bandas telúricas** sombreadas por especie (O₂ naranja, H₂O cian) con la **transmisión medida esa noche** en la tira de arriba, las **líneas de acreción** por familia (Balmer, He I, prohibidas, O I, Ca II, Paschen) y las **líneas de emisión de cielo** en gris discontinuo.

### Por qué bandas telúricas y no líneas telúricas

A la resolución de MUSE (FWHM ≈ 2.5 Å) las líneas individuales de O₂ y H₂O **no se resuelven**: dentro de un píxel espectral caen muchas. Marcar líneas sueltas daría una precisión que el dato no tiene, así que se marcan **bandas**. `molecfit` no está disponible aquí y, en estos datos, **no convergió** (A3 corrigió con la estrella telúrica estándar), pero de ahí quedó una **curva de transmisión medida** en la misma rejilla de λ: eso es más específico que cualquier lista de laboratorio y es lo que se dibuja. Catálogo y curva: [`musepipe/telluric_lines.py`](../musepipe/telluric_lines.py).

### Y sus datos, en columnas

La celda **escribe la tabla** además de la figura, en **ECSV** (el estándar portable de astropy): texto plano, con las unidades y la procedencia en la cabecera, que se lee con `Table.read(ruta)` sin configurar nada y se puede mandar por correo. Una figura sin sus datos no es un resultado citable.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from musepipe.paper_spectrum import (paper_spectrum_figure, pretty_flux_unit,
                                         spectrum_table_meta, write_spectrum_table)
    from musepipe.telluric_lines import measured_transmission
    from astropy.io import fits
    ROOT_P = nb.project_root()
    METHOD_P = 'lpm'
    PRODUCT_P = 'spec_lpm_object.fits'
    TARGET_P = (nb.run_target(RUN_ID) or RUN_ID).replace(' ', '')
    _h = fits.open(nb.run_dir(RUN_ID) / 'stages' / PRODUCT_P)
    _d = _h[1].data
    _cols = list(_d.columns.names)
    # La unidad viaja con el dato (BUNIT); no hay default silencioso.
    BUNIT_P = _h[1].header.get('BUNIT') or 'ADU'
    W_P = np.asarray(_d['wave_A'], float)
    F_P = np.asarray(_d['flux'], float)
    # El empírico manda; `flux_err` es el que eligió la etapa y solo
    # aporta algo cuando NO es el empírico (ver la nota de abajo).
    E_P = np.asarray(_d['flux_err_emp' if 'flux_err_emp' in _cols
                        else 'flux_err'], float)
    E_ALT_P = np.asarray(_d['flux_err'], float)
    EXTRA_P = {'flux_err_stat': E_ALT_P}
    for _c in ('apcorr', 'npix_eff', 'flags'):
        if _c in _cols:
            EXTRA_P[_c] = np.asarray(_d[_c])
    _h.close()
    try:
        MODO_P = (nb.load_qc('stages/spec_lpm_qc.json', RUN_ID).get('errors') or {}).get('mode')
    except Exception:
        MODO_P = None
    # Cuando la etapa eligió el error empírico, la columna `flux_err` ES
    # la empírica: dibujar las dos encima fingiría dos estimaciones
    # independientes donde solo hay una.
    if E_ALT_P is not None and np.allclose(E_ALT_P, E_P, equal_nan=True):
        E_ALT_P = None
        EXTRA_P.pop('flux_err_stat', None)
        print('las dos columnas de error coinciden (modo empírico):'
              ' una sola banda, y una sola columna en la tabla')
    # La transmisión telúrica MEDIDA de este run (A3). Si el objeto se
    # redujo en modo `cascade` no existe suelta: se marcan las bandas del
    # catálogo sin la profundidad de esa noche, y se dice.
    trans = measured_transmission(RUN_ID, project_root=ROOT_P)
    print('transmisión telúrica:', trans['source'] if trans else
          'no medida en este run — se marcan las bandas del catálogo')
    # Los canales que la etapa marcó como malos (hueco del láser AO) no
    # se dibujan: valen 0, y un 0 pintado se lee como una medida.
    from musepipe.extraction.aperture import FLAG_BAD_WINDOW
    MALOS_P = (np.asarray(EXTRA_P.get('flags', 0), dtype=int) & FLAG_BAD_WINDOW) != 0
    fig, _ejes = paper_spectrum_figure(
        W_P, F_P, E_P, flux_err_alt=E_ALT_P, bad_channels=MALOS_P,
        err_label='±1σ empírico (controles procesados igual)', err_alt_label='±1σ propagado del STAT (no es σ)',
        transmission=trans,
        title=nb.display_name(RUN_ID) + ' · ' + 'espectro del compañero · lpm (C6)',
        flux_label='flujo [' + pretty_flux_unit(BUNIT_P) + ']')
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'c6_lpm'
    outdir.mkdir(parents=True, exist_ok=True)
    # PDF además de PNG: es la que va al paper, y en vectorial las
    # etiquetas de las 24 líneas siguen leyéndose al ampliar. El PNG a
    # 300 dpi es el mínimo que piden las revistas para figuras de línea.
    DPI_P = 300      # súbelo si necesitas más resolución
    for ext in ('png', 'pdf'):
        fig.savefig(outdir / ('spectrum_paper' + '.' + ext), dpi=DPI_P)
    tabla = write_spectrum_table(
        nb.run_dir(RUN_ID) / 'tables' / ('spec_' + METHOD_P + '_' + TARGET_P + '.ecsv'),
        W_P, F_P, E_P, extra_columns=EXTRA_P,
        units={'flux': BUNIT_P, 'flux_err': BUNIT_P, 'flux_err_stat': BUNIT_P},
        meta=spectrum_table_meta(run_id=RUN_ID, target=TARGET_P, method=METHOD_P,
                                 product=PRODUCT_P, flux_unit=BUNIT_P,
                                 error_mode=MODO_P,
                                 extra={'figure': str(outdir / ('spectrum_paper' + '.pdf'))}))
    print('figura ->', outdir / ('spectrum_paper' + '.pdf'))
    print('tabla  ->', tabla, '(' + str(tabla.stat().st_size // 1024) + ' kB, '
          + str(int(np.size(W_P))) + ' canales)')
    print('        se lee con:  from astropy.table import Table; Table.read(ruta)')
    plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **Grado 4 congelado** (tres vías independientes del paper §3.2); diagnósticos de grado como QC con warnings, nunca ajuste al vuelo. · [`docs/spec_C6_codex_lpm_subtraction.md`](../docs/spec_C6_codex_lpm_subtraction.md)
- Máscara de líneas por target (`lpm_masked_lines_A`); una línea de ciencia sin enmascarar = auto-sustracción parcial silenciosa → el notebook la audita contra el catálogo G2.
- Sin pesos por varianza en v1 (OLS plano, como el paper); ponderación por precisión es trabajo futuro explícito. · [`docs/plan_integracion_halosub_julo2025.md`](../docs/plan_integracion_halosub_julo2025.md)


## Checks


In [ ]:
try:
    q = nb.load_qc('stages/spec_lpm_qc.json', RUN_ID)
    for k, v in q['checks'].items():
        print(f'  {k}: {v}')
    assert q['lpm']['degree'] == 4
except FileNotFoundError as e:
    print('QC aún no existe para este run:', e)


## Estado

**Pendiente de primera ejecución sobre datos reales** (checkpoint de la spec C6). Kernel verificado contra los oráculos del paper (Ec. B.5, límite R→0 de Fig. 2d) y contrato de etapa con tests sintéticos (2026-07-14).
